In [60]:
import numpy as np
import idx2numpy

In [81]:
class Network:
    
    def __init__(self, dimensions: tuple):
        self.dimensions = dimensions
        self.weights = []
        self.biases = []
        self.amount_of_layers = len(dimensions) #counted with input- and output-layer
        
        for i in range(1,len(dimensions)):
            self.biases.append(np.zeros(dimensions[i]))

        for j in range(0,len(dimensions)-1):
            r = dimensions[j]
            self.weights.append(np.random.randn(dimensions[j+1], r) * np.sqrt(2 / r))

    def forward_pass(self, input_data):
        if type(input_data) != np.ndarray:
            raise TypeError("input value must be of type np.ndarray")
        
        if len(input_data) != self.dimensions[0]:
            raise ValueError("input value must have the same length as defined in the network")

        self.activations = [input_data]
        aux_vector = input_data

        for i in range(len(self.weights) - 1):
            aux_vector = self.ReLU((self.weights[i] @ aux_vector) + self.biases[i])
            self.activations.append(aux_vector)

        output = self.softmax((self.weights[-1] @ aux_vector) + self.biases[-1])
        self.activations.append(output)

        return output
            
    def __str__(self):
        return "Neural Network: \n" + f"\tHidden Layers: {self.amount_of_layers-2}\n" + "\tDimensions " + " | ".join(str(dim) for dim in self.dimensions)

    def __repr__(self):
        return "Network"

    def ReLU(self, value):
        return np.maximum(0, value)

    def softmax(self, value):
        return np.exp(value)/sum(np.exp(value))

    def loss_function(self, predicted, expected):
        if len(predicted) != len(expected):
            raise ValueError("predicted and expected values must have the same length")
        
        if (type(predicted) != np.ndarray) or (type(expected) != np.ndarray):
            raise TypeError("predicted and expected values must be of type np.ndarray")

        return -np.sum(expected * np.log(predicted))
    
    def backward_pass(self, expected, learning_rate=0.15):
        L = len(self.weights)

        delta = self.activations[-1] - expected

        grads_W = [None] * L
        grads_b = [None] * L

        for k in reversed(range(L)):
            grads_W[k] = np.outer(delta, self.activations[k])
            grads_b[k] = delta

            if k > 0:
                relu_mask = (self.activations[k] > 0).astype(float)
                delta = (self.weights[k].T @ delta) * relu_mask

        for k in range(L):
            self.weights[k] -= learning_rate * grads_W[k]
            self.biases[k]  -= learning_rate * grads_b[k]


In [82]:
X_train = idx2numpy.convert_from_file("dataset/train-images.idx3-ubyte")
X_labels = idx2numpy.convert_from_file("dataset/train-labels.idx1-ubyte")
Y_eval = idx2numpy.convert_from_file("dataset/t10k-images.idx3-ubyte")
Y_labels = idx2numpy.convert_from_file("dataset/t10k-labels.idx1-ubyte")
print(X_train.shape)
print(X_labels.shape)
print(Y_eval.shape)
print(Y_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [83]:
#test to see if import worked
k = ""
for i in X_train[0]:
    for j in i:
        if j>0:
            k = k + f"X "
        else:
            k = k + f"  "
    print(k[:len(k)-1])
    k = ""

                                                       
                                                       
                                                       
                                                       
                                                       
                        X X X X X X X X X X X X        
                X X X X X X X X X X X X X X X X        
              X X X X X X X X X X X X X X X X          
              X X X X X X X X X X X                    
                X X X X X X X   X X                    
                  X X X X X                            
                      X X X X                          
                      X X X X                          
                        X X X X X X                    
                          X X X X X X                  
                            X X X X X X                
                              X X X X X                
                                  X X X X       

In [84]:
print(X_labels[0]==5)

True


In [89]:
#train the network
network = Network((784, 256, 128, 10))

for epoch in range(4):
    for i in range(len(X_train)):
        picture = X_train[i].flatten()
        y_sp = np.zeros(10)
        y_sp[X_labels[i]] = 1
        network.forward_pass(picture)
        network.backward_pass(y_sp)

C:\Users\joelz\AppData\Local\Temp\ipykernel_25744\1015295939.py:45: RuntimeWarning: overflow encountered in exp
  return np.exp(value)/sum(np.exp(value))
C:\Users\joelz\AppData\Local\Temp\ipykernel_25744\1015295939.py:45: RuntimeWarning: invalid value encountered in divide
  return np.exp(value)/sum(np.exp(value))


In [90]:
#evaluate the network

err_counter = 0

for j in range(len(Y_eval)):
    y_test = Y_eval[j].flatten()
    output = network.forward_pass(y_test)
    
    if output.argmax()+1 != Y_labels[j]:
        err_counter = err_counter + 1

print(f"Success rate: {((err_counter/10000)*100):.2f}%" )

Success rate: 88.65%


In [91]:
#evaluate the network

err_counter = 0

for j in range(len(X_train)):
    y_test = X_train[j].flatten()
    output = network.forward_pass(y_test)
    
    if output.argmax()+1 != X_labels[j]:
        err_counter = err_counter + 1

print(f"Success rate: {((err_counter/60000)*100):.2f}%" )

Success rate: 88.76%
